In [1]:
import time
import chromadb
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_core.documents import Document
from langchain_chroma import Chroma
from datetime import datetime


c:\AG0853_2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
pdf_folder_location = "tesla-annual-reports"

In [3]:
pdf_loader = PyPDFDirectoryLoader(pdf_folder_location)

In [4]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',
    chunk_size=512,
    chunk_overlap=16
)

In [5]:
tesla_10k_chunks = pdf_loader.load_and_split(text_splitter)

In [6]:
len(tesla_10k_chunks)

3337

In [7]:
print(tesla_10k_chunks[0])

page_content='UNITED	STATES
SECURITIES	AND	EXCHANGE	COMMISSION
Washington,	D.C.	20549
	
FORM	
10-K/A
(Amendment	No.	1)
	
(Mark	One)
☒
ANNUAL	REPORT	PURSUANT	TO	SECTION	13	OR	15(d)	OF	THE	SECURITIES	EXCHANGE	ACT	OF	1934
For	the	fiscal	year	ended	
December	31,	
2021
	
OR
☐
TRANSITION	REPORT	PURSUANT	TO	SECTION	13	OR	15(d)	OF	THE	SECURITIES	EXCHANGE	ACT	OF	1934
For	the	transition	period	from	
																				
	to	
																					
Commission	File	Number:	
001-34756
	
Tesla,	Inc.
(Exact	name	of	registrant	as	specified	in	its	charter)
	
	
Delaware
	
91-2197729
(State	or	other	jurisdiction	of
incorporation	or	organization)
	
(I.R.S.	Employer
Identification	No.)
	
	
	
1	Tesla	Road
Austin
,	
Texas
	
78725
(Address	of	principal	executive	offices)
	
(Zip	Code)
(
512
)	
516-8177
(Registrant’s	telephone	number,	including	area	code)
Securities	registered	pursuant	to	Section	12(b)	of	the	Act:
	
Title	of	each	class
Trading	Symbol(s)
Name	of	each	exchange	on	which	registered
Common	stock
TSLA

### Database Creation

In [8]:
tesla_10k_collection = 'tesla-10k-2019-to-2023'

In [9]:
from langchain_community.embeddings import HuggingFaceEmbeddings

In [10]:
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

C:\Users\Priyanshi Garg\AppData\Local\Temp\ipykernel_2172\4093584008.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")


In [11]:
chromadb_client = chromadb.PersistentClient(
    path="./tesla_db"
)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


In [13]:
chromadb_client.heartbeat()

1780377646865925600

In [14]:
chromadb_client.count_collections()

1

In [15]:
vectorstore = Chroma(
    collection_name=tesla_10k_collection,
    collection_metadata={"hnsw:space": "cosine"},
    embedding_function=embedding,
    client=chromadb_client,
    persist_directory="./tesla_db"
)

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [16]:
chromadb_client.count_collections()

1

In [17]:
chromadb_client.list_collections()

['tesla-10k-2019-to-2023']

In [ ]:
# i = 0 # Initialize the starting index for the chunks

# while i < len(tesla_10k_chunks): # Iterate while the index is less than the total number of chunks
#     vectorstore.add_documents( # Add documents to the vector store in batches of 500
#         documents=tesla_10k_chunks[i:i+500], # Get the current batch of 500 chunks
#         ids=["text_" + str(i) for i in range(i, i+500)] # Assign unique IDs to each chunk in the batch
#     )

#     i += 500 # Increment the index by 500 to move to the next batch
#     time.sleep(15) # Pause for 30 seconds to avoid rate limiting issues with the vector store

In [18]:
from dotenv import load_dotenv
load_dotenv()
import os
os.environ['GEMINI_API_KEY'] = os.getenv('GEMINI_API_KEY')

In [19]:
from openai import OpenAI
client = OpenAI(
    api_key=os.environ['GEMINI_API_KEY'],
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [20]:
model = "gemini-2.5-flash"

In [21]:
vectorstore_persisted = Chroma(
    collection_name=tesla_10k_collection,
    collection_metadata={"hnsw:space": "cosine"},
    embedding_function=embedding,
    client=chromadb_client,
    persist_directory="./tesla_db"
)

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [22]:
retriever = vectorstore_persisted.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 5}
)

In [23]:
qna_system_message = """
You are an assistant to a financial services firm who answers user queries on annual reports.
User input will have the context required by you to answer user queries.
This context will be delimited by: <Context> and </Context>.
The context contains references to specific portions of a document relevant to the user query.

User queries will be delimited by: <Question> and </Question>.

Please answer user queries only using the context provided in the input.
Do not mention anything about the context in your final answer. Your response should only contain the answer to the question.

If the answer is not found in the context, respond "I don't know".
"""

In [24]:
qna_user_message_template = """
<Context>
Here are some documents that are relevant to the question mentioned below.
{context}
</Context>

<Question>
{question}
</Question>
"""

In [25]:
user_query = "What was the automotive revenue in 2021?"

In [26]:
relevant_document_chunks = retriever.invoke(user_query)

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


In [27]:
len(relevant_document_chunks)

5

In [28]:
for document in relevant_document_chunks:
    print(document)
    break

page_content='2022
2021
$
%
$
%
Automotive	sales
$
78,509	
$
67,210	
$
44,125	
$
11,299	
17	
%
$
23,085	
52	
%
Automotive	regulatory	credits
1,790	
1,776	
1,465	
14	
1	
%
311	
21	
%
Automotive	leasing
2,120	
2,476	
1,642	
(356)
(14)
%
834	
51	
%
Total	automotive	revenues
82,419	
71,462	
47,232	
10,957	
15	
%
24,230	
51	
%
Services	and	other
8,319	
6,091	
3,802	
2,228	
37	
%
2,289	
60	
%
Total	automotive	&	services	and	other	segment
revenue
90,738	
77,553	
51,034	
13,185	
17	
%
26,519	
52	
%
Energy	generation	and	storage	segment	revenue
6,035	
3,909	
2,789	
2,126	
54	
%
1,120	
40	
%
Total	revenues
$
96,773	
$
81,462	
$
53,823	
$
15,311	
19	
%
$
27,639	
51	
%
Automotive	&	Services	and	Other	Segment
Automotive	sales	revenue	includes	revenues	related	to	cash	and	financing	deliveries	of	new	Model	S,	Model	X,	Semi,	Model	3,	Model	Y,	and
Cybertruck	vehicles,	including	access	to	our	FSD	Capability	features	and	their	ongoing	maintenance,	internet	connectivity,	free	Supercharging	programs
and	ov

In [29]:


relevant_document_chunks = retriever.invoke(user_query)
context_list = [d.page_content for d in relevant_document_chunks]
context_for_query = "\n---\n".join(context_list)

prompt = [
    {'role': 'system', 'content': qna_system_message},
    {'role': 'user', 'content': qna_user_message_template.format(
         context=context_for_query,
         question=user_query
        )
    }
]


try:
    response = client.chat.completions.create(
        model=model,
        messages=prompt,
        temperature=0
    )

    prediction = response.choices[0].message.content.strip()
except Exception as e:
    prediction = f'Sorry, I encountered the following error: \n {e}'

print(prediction)

Sorry, I encountered the following error: 
 Error code: 503 - [{'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}]


### Query Expansion

In [30]:
query_expansion_system_message = """
You are an financial domain expert assisting in answering questions related to 10-k reports.
Perform query expansion on the question below. If there are multiple common ways of phrasing a user question 
or common synonyms for key words in the question, make sure to return multiple versions \
of the query with the different phrasings.

If there are acronyms or words you are not familiar with, do not try to rephrase them.

Return at least 3 versions of the question as a list.
Generate only a list of questions, each question in a new line.
Do not number the list of questions or use bullet points.
Do not mention anything before or after the list.
"""

user_message_template="""
<Question>
{question}
</Question>
"""

In [31]:
user_input = "What was the automotive revenue in 2021?"

In [32]:
prompt = [
    {'role':'system', 'content': query_expansion_system_message},
    {'role': 'user', 'content': user_message_template.format(
        question = user_input
       )
    }
]

In [33]:
prompt

[{'role': 'system',
  'content': '\nYou are an financial domain expert assisting in answering questions related to 10-k reports.\nPerform query expansion on the question below. If there are multiple common ways of phrasing a user question \nor common synonyms for key words in the question, make sure to return multiple versions of the query with the different phrasings.\n\nIf there are acronyms or words you are not familiar with, do not try to rephrase them.\n\nReturn at least 3 versions of the question as a list.\nGenerate only a list of questions, each question in a new line.\nDo not number the list of questions or use bullet points.\nDo not mention anything before or after the list.\n'},
 {'role': 'user',
  'content': '\n<Question>\nWhat was the automotive revenue in 2021?\n</Question>\n'}]

In [34]:
query_expansions = client.chat.completions.create(model=model,
                                                  messages=prompt,
                                                  temperature=0)

In [35]:
print(query_expansions.choices[0].message.content)

What was the automotive sales in 2021?
In 2021, what was the automotive revenue?
Could you tell me the automotive revenue for 2021?
What was the total automotive revenue in 2021?


In [36]:
print(query_expansions)

ChatCompletion(id='aYYdaqvYIO-Fg8UP4cr8yA4', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='What was the automotive sales in 2021?\nIn 2021, what was the automotive revenue?\nCould you tell me the automotive revenue for 2021?\nWhat was the total automotive revenue in 2021?', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None))], created=1780319852, model='gemini-2.5-flash', object='chat.completion', service_tier=None, system_fingerprint=None, usage=CompletionUsage(completion_tokens=55, prompt_tokens=168, total_tokens=427, completion_tokens_details=None, prompt_tokens_details=None))


In [37]:
query_expansions_list = query_expansions.choices[0].message.content.strip().split('\n')


In [38]:
query_expansions_list

['What was the automotive sales in 2021?',
 'In 2021, what was the automotive revenue?',
 'Could you tell me the automotive revenue for 2021?',
 'What was the total automotive revenue in 2021?']

In [39]:
expanded_context_list = []

In [40]:
for query in query_expansions_list:
    expanded_context_list.extend([d.page_content for d in retriever.invoke(query)])

In [41]:
expanded_context_list

['2022\n2021\n$\n%\n$\n%\nAutomotive\tsales\n$\n78,509\t\n$\n67,210\t\n$\n44,125\t\n$\n11,299\t\n17\t\n%\n$\n23,085\t\n52\t\n%\nAutomotive\tregulatory\tcredits\n1,790\t\n1,776\t\n1,465\t\n14\t\n1\t\n%\n311\t\n21\t\n%\nAutomotive\tleasing\n2,120\t\n2,476\t\n1,642\t\n(356)\n(14)\n%\n834\t\n51\t\n%\nTotal\tautomotive\trevenues\n82,419\t\n71,462\t\n47,232\t\n10,957\t\n15\t\n%\n24,230\t\n51\t\n%\nServices\tand\tother\n8,319\t\n6,091\t\n3,802\t\n2,228\t\n37\t\n%\n2,289\t\n60\t\n%\nTotal\tautomotive\t&\tservices\tand\tother\tsegment\nrevenue\n90,738\t\n77,553\t\n51,034\t\n13,185\t\n17\t\n%\n26,519\t\n52\t\n%\nEnergy\tgeneration\tand\tstorage\tsegment\trevenue\n6,035\t\n3,909\t\n2,789\t\n2,126\t\n54\t\n%\n1,120\t\n40\t\n%\nTotal\trevenues\n$\n96,773\t\n$\n81,462\t\n$\n53,823\t\n$\n15,311\t\n19\t\n%\n$\n27,639\t\n51\t\n%\nAutomotive\t&\tServices\tand\tOther\tSegment\nAutomotive\tsales\trevenue\tincludes\trevenues\trelated\tto\tcash\tand\tfinancing\tdeliveries\tof\tnew\tModel\tS,\tModel\tX,\tSem

In [42]:
final_context_documents = set(expanded_context_list)

In [43]:
len(final_context_documents)

7

In [44]:
final_context_documents

{'(Dollars\tin\tmillions)\n2023\n2022\n2021\n$\n%\n$\n%\nCost\tof\trevenues\nAutomotive\tsales\n$\n65,121\t\n$\n49,599\t\n$\n32,415\t\n$\n15,522\t\n31\t\n%\n$\n17,184\t\n53\t\n%\nAutomotive\tleasing\n1,268\t\n1,509\t\n978\t\n(241)\n(16)\n%\n531\t\n54\t\n%\nTotal\tautomotive\tcost\tof\trevenues\n66,389\t\n51,108\t\n33,393\t\n15,281\t\n30\t\n%\n17,715\t\n53\t\n%\nServices\tand\tother\n7,830\t\n5,880\t\n3,906\t\n1,950\t\n33\t\n%\n1,974\t\n51\t\n%\nTotal\tautomotive\t&\tservices\tand\tother\tsegment\ncost\tof\trevenues\n74,219\t\n56,988\t\n37,299\t\n17,231\t\n30\t\n%\n19,689\t\n53\t\n%\nEnergy\tgeneration\tand\tstorage\tsegment\n4,894\t\n3,621\t\n2,918\t\n1,273\t\n35\t\n%\n703\t\n24\t\n%\nTotal\tcost\tof\trevenues\n$\n79,113\t\n$\n60,609\t\n$\n40,217\t\n$\n18,504\t\n31\t\n%\n$\n20,392\t\n51\t\n%\nGross\tprofit\ttotal\tautomotive\n$\n16,030\t\n$\n20,354\t\n$\n13,839\t\nGross\tmargin\ttotal\tautomotive\n19.4\t\n%\n28.5\t\n%\n29.3\t\n%\nGross\tprofit\ttotal\tautomotive\t&\tservices\tand\tothe

In [45]:
system_message = """
You are a helpful AI assistant.
Answer only from the provided context.
If the answer is not in the context, say so.
"""

user_input = "What was the automotive revenue in 2021?"

context = "\n\n".join(final_context_documents)

prompt = f"""

Context:
{context}

Question:
{user_input}

Answer:
"""

print(prompt)



Context:
almost	entirely	offset	by	our	cost	savings	initiatives	and	payroll	related	benefits.
	
Revenues
	
	
	
Year	Ended	December	31,
	
	
2020
	
vs.
	
2019
	
Change
	
	
2019
	
vs.
	
2018
	
Change
	
(Dollars	in	millions)
	
2020
	
	
2019
	
	
2018
	
	
$
	
	
%
	
	
$
	
	
%
	
Automotive	sales
	
$
26,184
	
	
$
19,952
	
	
$
17,632
	
	
$
6,232
	
	
	
31
%
	
$
2,320
	
	
	
13
%
Automotive	leasing
	
	
1,052
	
	
	
869
	
	
	
883
	
	
	
183
	
	
	
21
%
	
	
(14
)
	
	
-2
%
Total	automotive	revenues
	
	
27,236
	
	
	
20,821
	
	
	
18,515
	
	
	
6,415
	
	
	
31
%
	
	
2,306
	
	
	
12
%
Services	and	other
	
	
2,306
	
	
	
2,226
	
	
	
1,391
	
	
	
80
	
	
	
4
%
	
	
835
	
	
	
60
%
Total	automotive	&	services	and	other
			segment	revenue
	
	
29,542
	
	
	
23,047
	
	
	
19,906
	
	
	
6,495
	
	
	
28
%
	
	
3,141
	
	
	
16
%

November	2021	for	additional	taxes	owed,	if	any.
	
Tesla	periodically	does	business	with	certain	entities	with	which	its	CEO	and	directors	are	affiliated,	such	as	SpaceX	and	Twitter,	Inc.,	in
	
accordan

In [46]:
response = client.chat.completions.create(
    model=model,
    messages=[
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
    ]
)

print(response.choices[0].message.content)

Automotive revenue in 2021 was $47,232 million.


### Hypothetical Questions

In [23]:
BATCH_SIZE = 20

batches = [
    tesla_10k_chunks[i:i + BATCH_SIZE]
    for i in range(0, len(tesla_10k_chunks), BATCH_SIZE)
]

print(f"Total chunks: {len(tesla_10k_chunks)}")
print(f"Total batches: {len(batches)}")
print(f"First batch size: {len(batches[0])}")
print(f"Last batch size: {len(batches[-1])}")

Total chunks: 3337
Total batches: 167
First batch size: 20
Last batch size: 17


In [25]:
hypothetical_questions_system_message = """
You will receive multiple document chunks.

For EACH document chunk:
1. Generate exactly 3 hypothetical questions that could be answered using the information in that chunk.
2. Questions should be realistic user questions.
3. Questions should be specific enough to retrieve the chunk during semantic search.
4. Do not copy text verbatim from the chunk.
5. Do not generate more or fewer than 3 questions.

Return ONLY valid JSON.

Output format:

[
  {
    "chunk_id": "<chunk_id>",
    "questions": [
      "question_1",
      "question_2",
      "question_3"
    ]
  }
]
""" 

In [ ]:
user_message_template = """
Generate hypothetical questions for the following document chunks.

{documents}
"""

In [ ]:
import json

documents_payload = []

for doc in batches:

    documents_payload.append({
        "chunk_id": str(doc.id),
        "content": doc.page_content
    })

user_prompt = user_message_template.format(
    documents=json.dumps(documents_payload, ensure_ascii=False)
)

hypothetical_questions_system_message = """
Generate a list of exactly 3 hypothetical questions that the document presented in the input could be used to answer.
Generate only a list of questions, each question in a new line.
Do not number the questions or use bullet points.
Do not mention anything before or after the list.
"""

user_message_template = """
<Document>
{document}
</Document>
"""

In [ ]:
hypothetical_questions = []

In [ ]:
tesla_10k_chunks

In [ ]:
for document in tesla_10k_chunks:

    try:
        response = client.chat.completions.create(
        model=model,
        messages=[
        {"role": "system", "content": hypothetical_questions_system_message},
        {"role": "user", "content": user_message_template.format(document=document)}
        ], temperature=0
    )


        questions = response.choices[0].message.content.strip()
    except Exception as e:  
        print(e)
        questions = ""

    questions_list = questions.split("\n")

    for question in questions_list:

        questions_metadata = {
            'parent_chunk_id': document.id,
            'parent_collection': 'full_document_chunks'
        }

        hypothetical_questions.append(
            Document(
                page_content=question,
                metadata=questions_metadata
            )
        )

In [ ]:
hypothetical_questions

In [ ]:
len(hypothetical_questions)

In [ ]:
hypothetical_questions[0], hypothetical_questions[1], hypothetical_questions[2]

In [ ]:
hypothetical_questions_vectorstore = Chroma(
    collection_name="hypothetical_questions",
    collection_metadata={"hnsw:space": "cosine"},
    embedding_function=embedding_model,
    client=chromadb_client
)

In [ ]:
chromadb_client.list_collections()

In [ ]:
hypothetical_questions_vectorstore.add_documents(
    documents=hypothetical_questions
)

In [ ]:
retriever = hypothetical_questions_vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={'k': 5}
)

In [ ]:
user_query = "Any specific fines levied on the company in 2022?"

In [ ]:
hypothetical_questions_retrieved = retriever.invoke(user_query)

In [ ]:
hypothetical_questions_retrieved

In [ ]:
retrived_documents = vectorstore.get(
    ids=list(set([d.metadata['parent_chunk_id'] for d in hypothetical_questions_retrieved]))
)

In [ ]:
print(retrived_documents)

In [ ]:
retrived_documents['documents']

In [ ]:
len(retrived_documents['documents'])